# Cross-Component Interaction

How attention and MLP interact within layers: output alignment,
logit interaction, cancellation, and dominance patterns.

## Why This Matters

Cross-component component interaction measures how information transfers between different parts of the model. Understanding cross-component interactions reveals the collaborative computation that produces emergent model capabilities.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.cross_component_interaction import (
    attn_mlp_output_alignment, attn_mlp_logit_interaction,
    component_cancellation, component_dominance_pattern,
    cross_component_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Attention-MLP Output Alignment

Do attention and MLP reinforce or contradict each other?

In [ ]:
for layer in range(4):
    result = attn_mlp_output_alignment(model, tokens, layer=layer)
    flag = ' [COOPERATIVE]' if result['is_cooperative'] else ''
    print(f"  Layer {layer}: align={result['mean_alignment']:.4f}{flag}")

## Logit Interaction

Do attention and MLP agree on token predictions?

In [ ]:
for layer in range(4):
    result = attn_mlp_logit_interaction(model, tokens, layer=layer, position=-1)
    flag = ' [AGREE]' if result['predictions_agree'] else ' [DISAGREE]'
    print(f"  Layer {layer}: attn→{result['attn_prediction']:3d}, "
          f"mlp→{result['mlp_prediction']:3d}, corr={result['logit_correlation']:.4f}{flag}")

## Component Cancellation

Do attention and MLP partially cancel each other?

In [ ]:
for layer in range(4):
    result = component_cancellation(model, tokens, layer=layer, position=-1)
    flag = ' [CANCELLATION]' if result['has_cancellation'] else ''
    print(f"  Layer {layer}: cancel={result['cancellation_fraction']:.4f}, "
          f"a={result['attn_norm']:.4f}, m={result['mlp_norm']:.4f}, "
          f"combined={result['combined_norm']:.4f}{flag}")

## Dominance Pattern

Which component dominates at each layer?

In [ ]:
result = component_dominance_pattern(model, tokens)
print(f"Attention dominant in {result['attn_dominant_count']} layers\n")
for p in result['per_layer']:
    attn_bar = '█' * int(p['attn_norm'] * 10)
    mlp_bar = '█' * int(p['mlp_norm'] * 10)
    print(f"  Layer {p['layer']}: [{p['dominant']:9s}] attn={attn_bar} mlp={mlp_bar}")

## Cross-Component Summary

In [ ]:
result = cross_component_summary(model, tokens)
for p in result['per_layer']:
    flag = ' [COOP]' if p['is_cooperative'] else ''
    print(f"  Layer {p['layer']}: align={p['alignment']:.4f}, "
          f"a={p['attn_norm']:.4f}, m={p['mlp_norm']:.4f}, "
          f"{p['dominant']}{flag}")